# 2026 Toronto election: census-tract GeoPackages

Builds two GeoPackages in `data/election_results/2026-results-clean/`:

1. **`mayor_2023_2026_ct.gpkg`** — mayoral results for 2023 and 2026 on the 585 Toronto 2021 census tracts (CTs), joined to the 2021 census profile. 2023 keeps Chow, Bailão, Saunders, Furey, Hunter, Brown and Bradford plus an "other" bucket; 2026 keeps Chow and Bradford plus "other".
2. **`councillor_2026_ct.gpkg`** — 2026 city councillor results on CT×ward pieces. A council race only exists inside a ward, so a CT that straddles a ward boundary is split into one piece per ward (unless ≥95% of it sits in one ward). Census counts on split CTs are divided by area share.

**Sources**

| What | Where |
|---|---|
| CT-level mayoral votes 1997–2026 | `data/election_results/2026-results-clean/tor_electoral_ct2021_pct.dta` |
| 2026 poll-by-poll (mayor, councillor) | `data/election_results/2026-results-raw/Poll_By_Poll_*.xlsx` |
| 2026 voting-subdivision polygons | `data/election_results/2026-results-raw/voting_subdivisions.geojson` (City of Toronto `cot_geospatial6/MapServer/10`) |
| 2021 CT and DA boundaries, DA citizen-adult counts | `data/toronto_election_turnout/census/`, `analysis/toronto_election_turnout/data/archive/census/processed/da/` |
| 25-ward boundaries | `src/data/wards.geo.json` |
| 2021 census profile (+ labels) | `src/routes/social-geography-and-the-mayoral-vote/data/` |

In [1]:
import json
import re
import warnings
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio
from shapely import STRtree, area, intersection, make_valid, union_all
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

In [2]:
REPO_ROOT = Path.cwd().resolve().parents[1]
ELECTION_DIR = REPO_ROOT / "data" / "election_results"
RAW_DIR = ELECTION_DIR / "2026-results-raw"
OUT_DIR = ELECTION_DIR / "2026-results-clean"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DTA_PATH = OUT_DIR / "tor_electoral_ct2021_pct.dta"
MAYOR_XLSX = RAW_DIR / "Poll_By_Poll_Mayor.xlsx"
COUNCILLOR_XLSX = RAW_DIR / "Poll_By_Poll_Councillor.xlsx"
VSD_PATH = RAW_DIR / "voting_subdivisions.geojson"

CT_PATH = REPO_ROOT / "data" / "toronto_election_turnout" / "census" / "ct_census_profile.geojson"
DA_PATH = (REPO_ROOT / "analysis" / "toronto_election_turnout" / "data" / "archive" / "census"
           / "processed" / "da" / "statcan_2021_toronto_da.geojson")
WARD_PATH = REPO_ROOT / "src" / "data" / "wards.geo.json"
CENSUS_DIR = REPO_ROOT / "src" / "routes" / "social-geography-and-the-mayoral-vote" / "data"
CENSUS_PATH = CENSUS_DIR / "census.parquet"
CENSUS_META_PATH = CENSUS_DIR / "meta.json"

MAYOR_GPKG = OUT_DIR / "mayor_2023_2026_ct.gpkg"
COUNCILLOR_GPKG = OUT_DIR / "councillor_2026_ct.gpkg"
DICTIONARY_CSV = OUT_DIR / "census_variable_dictionary.csv"

WORKING_EPSG = 3347    # StatCan Lambert (metres) for all area arithmetic
OUTPUT_EPSG = 4326     # CRS the GeoPackages are written in
CENSUS_YEAR = 2021
ANCILLARY_WEIGHT_FIELD = "citizen_canadian_18over"   # DA weight used to place voters inside a polygon

# Mayoral candidates kept by name: slug (as in the .dta columns a_<slug><year>) -> label
MAYOR_CANDIDATES = {
    2023: {
        "chow_olivia": "Olivia Chow", "bailao_ana": "Ana Bailão", "saunders_mark": "Mark Saunders",
        "furey_anthony": "Anthony Furey", "hunter_mitzie": "Mitzie Hunter", "brown_chloe": "Chloe Brown",
        "bradford_brad": "Brad Bradford",
    },
    2026: {"chow_olivia": "Olivia Chow", "bradford_brad": "Brad Bradford"},
}

# Ward splitting rules
UNSPLIT_THRESHOLD = 0.95     # a CT with >=95% of its (in-ward) area in one ward is not split
MIN_PIECE_SHARE = 0.05       # in a split CT, a ward holding <5% of the CT gets no piece (merged into the largest piece)

# Poll-by-poll workbook layout: 97/98/99 are reporting buckets (advance, institutions, ...) with no polygon.
# tor_electoral_ct2021_pct.dta leaves them out of its 2026 mayoral allocation, so the default here does too.
SPECIAL_BUCKETS = (97, 98, 99)
INCLUDE_SPECIAL_BUCKETS = False   # True: spread each ward's 97-99 votes over that ward's pieces (population-weighted)

TOLERANCE = 1e-6

## 1. Load boundaries

All geometries go to `EPSG:3347` and through `make_valid` (StatCan and City polygons contain small self-intersections that would otherwise give wrong intersection areas). The CT universe is the 585 Toronto CTs that contain DAs; the CT key `ctuid2021` is the `"5350001.00"` string the `.dta` and census table use.

In [3]:
def load_valid(path, to_epsg=WORKING_EPSG):
    gdf = gpd.read_file(path).to_crs(to_epsg).reset_index(drop=True)
    gdf["geometry"] = gdf.geometry.apply(make_valid)
    return gdf

ct = load_valid(CT_PATH)
ct = ct[ct["in_interpolation_universe"] == True].reset_index(drop=True)  # noqa: E712
ct["ctuid2021"] = ct["ct_id"].map(lambda x: f"{x:.2f}")
ct = ct[["ctuid2021", "geometry"]]
assert len(ct) == 585 and ct["ctuid2021"].is_unique

da = load_valid(DA_PATH)
da["ctuid2021"] = da["ct_id"].astype(float).map(lambda x: f"{x:.2f}")
da["area_m2"] = da.geometry.area
da["weight"] = da[ANCILLARY_WEIGHT_FIELD].fillna(0.0)   # suppressed DA counts -> zero weight, not dropped
da = da[["ctuid2021", "area_m2", "weight", "geometry"]]
assert da["ctuid2021"].isin(ct["ctuid2021"]).all()

wards = load_valid(WARD_PATH).rename(columns={"num": "ward", "name": "ward_name"})
wards["ward"] = wards["ward"].astype(int)
wards = wards[["ward", "ward_name", "geometry"]].sort_values("ward").reset_index(drop=True)
assert len(wards) == 25

vsd = load_valid(VSD_PATH)
vsd["ward"] = vsd["AREA_LONG_CODE"].str[:2].astype(int)
vsd["sub"] = vsd["AREA_LONG_CODE"].str[2:].astype(int)
vsd["poll_id"] = vsd["AREA_LONG_CODE"]
vsd = vsd[["poll_id", "ward", "sub", "geometry"]]
assert vsd["poll_id"].is_unique

print(f"{len(ct)} CTs, {len(da)} DAs ({(da['weight'] == 0).sum()} with zero weight), {len(wards)} wards, {len(vsd)} voting subdivisions")

585 CTs, 3743 DAs (16 with zero weight), 25 wards, 1356 voting subdivisions


**Check: does each voting-subdivision polygon sit inside the ward its code names?** The ward layer and the subdivision layer come from different City files, so confirm they agree before using the ward as a hard constraint.

In [4]:
ward_geom = wards.set_index("ward").geometry
inside = np.array([
    area(intersection(g, ward_geom[w])) / area(g) for g, w in zip(vsd.geometry.values, vsd["ward"].values)
])
print("share of each subdivision lying in its own ward:", pd.Series(inside).describe().round(4).to_dict())
print("subdivisions with <90% in own ward:", int((inside < 0.9).sum()))

share of each subdivision lying in its own ward: {'count': 1356.0, 'mean': 0.9999, 'std': 0.0004, 'min': 0.9875, '25%': 0.9999, '50%': 1.0, '75%': 1.0, 'max': 1.0}
subdivisions with <90% in own ward: 0


## 2. Parse the poll-by-poll workbooks

One sheet per ward. Layout: a `City Ward N <name>` row, a `Subdivision` row of subdivision numbers ending in `Total`, a race-title row, one row per candidate, and a `City Ward N Totals` row. Output is a long table `ward, sub, candidate, votes`. Two assertions guard the parse: each subdivision's candidate rows must add up to the sheet's own totals row, and each candidate's subdivisions must add up to the sheet's `Total` column.

In [5]:
def parse_poll_by_poll(path):
    rows = []
    for sheet, x in pd.read_excel(path, sheet_name=None, header=None).items():
        ward = int(sheet.split()[1])
        subs = x.iloc[1, 1:-1].astype(int).tolist()
        totals_row = x.iloc[-1, 1:-1].astype(float).values
        body = x.iloc[3:-1]
        body = body[body[0].notna()]
        votes = body.iloc[:, 1:-1].astype(float).values
        assert np.allclose(votes.sum(axis=0), totals_row), f"{path.name} {sheet}: subdivision totals disagree"
        assert np.allclose(votes.sum(axis=1), body.iloc[:, -1].astype(float).values), f"{path.name} {sheet}: candidate totals disagree"
        for name, v in zip(body[0], votes):
            rows.extend((ward, s, name.strip(), n) for s, n in zip(subs, v))
    return pd.DataFrame(rows, columns=["ward", "sub", "candidate", "votes"])

mayor_polls = parse_poll_by_poll(MAYOR_XLSX)
council_polls = parse_poll_by_poll(COUNCILLOR_XLSX)
for df in (mayor_polls, council_polls):
    df["poll_id"] = df["ward"].map("{:02d}".format) + df["sub"].map("{:03d}".format)

unmatched = set(council_polls["poll_id"]) - set(vsd["poll_id"])
assert {int(p[2:]) for p in unmatched} <= set(SPECIAL_BUCKETS), "unexpected subdivisions without a polygon"
assert set(vsd["poll_id"]) <= set(council_polls["poll_id"])

bucket_share = council_polls.loc[council_polls["sub"].isin(SPECIAL_BUCKETS), "votes"].sum() / council_polls["votes"].sum()
bucket_share_mayor = mayor_polls.loc[mayor_polls["sub"].isin(SPECIAL_BUCKETS), "votes"].sum() / mayor_polls["votes"].sum()
print(f"councillor: {council_polls['candidate'].nunique()} candidate names over 25 wards, {council_polls['votes'].sum():,.0f} votes; "
      f"{bucket_share:.1%} sit in subdivisions {SPECIAL_BUCKETS} (no polygon)")
print(f"mayor: {mayor_polls['votes'].sum():,.0f} votes; {bucket_share_mayor:.1%} in subdivisions {SPECIAL_BUCKETS}")

councillor: 190 candidate names over 25 wards, 932,157 votes; 24.0% sit in subdivisions (97, 98, 99) (no polygon)
mayor: 982,660 votes; 23.6% in subdivisions (97, 98, 99)


## 3. Population-weighted areal interpolation

The same method as `analysis/toronto_election_turnout/04_interpolate_votes_to_tracts.ipynb`. For one source polygon (a voting subdivision):

1. Intersect it with every target (CT, or CT×ward piece).
2. Inside each target-piece, intersect with the DAs belonging to that CT and weight each DA fragment by `citizen_canadian_18over × (fragment area / DA area)`, so votes follow where adult citizens live rather than land area.
3. Normalise the target weights to sum to 1 across all targets the polygon touches, and multiply the polygon's votes by them. Votes are conserved by construction.

If every DA under a polygon has zero weight, the weights fall back to area. If a polygon touches no target at all (only possible when targets are restricted to its ward), the votes go to the nearest target in the ward.

`match_ward=True` restricts a polygon to targets in its own ward — needed for councillor votes, which only exist inside a ward.

In [6]:
da_tree = STRtree(da.geometry.values)
da_ct = da["ctuid2021"].values
da_geom = da.geometry.values
da_area = da["area_m2"].values
da_weight = da["weight"].values


def build_crosswalk(sources, targets, match_ward, allow_area_fallback=True, desc="crosswalk"):
    '''Source polygon -> target allocation weights.

    sources: DataFrame[src_id, ward, geometry]; targets: DataFrame[tgt_id, ctuid2021, ward, geometry].
    Returns DataFrame[src_id, tgt_id, weight] (weights sum to 1 per src_id) and a diagnostics dict.
    '''
    tgt_ids = targets["tgt_id"].values
    tgt_ct = targets["ctuid2021"].values
    tgt_geom = targets.geometry.values
    groups = {}
    if match_ward:
        for w, idx in targets.groupby("ward").indices.items():
            groups[w] = (idx, STRtree(tgt_geom[idx]))
    else:
        idx = np.arange(len(targets))
        groups[None] = (idx, STRtree(tgt_geom))

    rows, n_area_fallback, n_nearest, n_dropped = [], 0, 0, 0
    for src_id, ward, geom in tqdm(zip(sources["src_id"], sources["ward"], sources.geometry.values),
                                   total=len(sources), desc=desc):
        idx, tree = groups[ward if match_ward else None]
        overlaps, total_pop, total_area = [], 0.0, 0.0
        for j in tree.query(geom, predicate="intersects"):
            t = idx[j]
            piece = intersection(geom, tgt_geom[t])
            piece_area = area(piece)
            if piece_area <= 0:
                continue
            pop = 0.0
            for d in da_tree.query(piece, predicate="intersects"):
                if da_ct[d] != tgt_ct[t] or da_area[d] <= 0:
                    continue
                pop += da_weight[d] * area(intersection(piece, da_geom[d])) / da_area[d]
            overlaps.append((tgt_ids[t], pop, piece_area))
            total_pop += pop
            total_area += piece_area
        if not overlaps:
            nearest = tree.query_nearest(geom)
            rows.append((src_id, tgt_ids[idx[nearest[0]]], 1.0))
            n_nearest += 1
            continue
        use_area = total_pop <= 0
        if use_area and not allow_area_fallback:
            n_dropped += 1
            continue
        n_area_fallback += use_area
        denom = total_area if use_area else total_pop
        rows.extend((src_id, tid, (a if use_area else p) / denom) for tid, p, a in overlaps)
    xw = pd.DataFrame(rows, columns=["src_id", "tgt_id", "weight"])
    return xw, {"area_fallback": n_area_fallback, "nearest_fallback": n_nearest, "dropped": n_dropped}


def allocate(xw, votes, key="src_id"):
    '''Spread source-level votes (src_id, candidate, votes) onto targets through the crosswalk.'''
    j = xw.merge(votes, on="src_id")
    j["alloc"] = j["weight"] * j["votes"]
    return j.groupby(["tgt_id", "candidate"])["alloc"].sum().unstack(fill_value=0.0)

## 4. Validate the method against the `.dta`

Re-derive 2026 mayoral votes per whole CT with this crosswalk and compare with `a_chow_olivia2026` / `a_bradford_brad2026` in `tor_electoral_ct2021_pct.dta`. The `.dta` allocates only subdivisions that have a polygon (it excludes the 97–99 buckets), so this check does the same.

In [7]:
dta = pd.read_stata(DTA_PATH).set_index("ctuid2021")
assert set(dta.index) == set(ct["ctuid2021"])

ct_targets = ct.assign(tgt_id=ct["ctuid2021"], ward=0)
poll_sources = vsd.assign(src_id=vsd["poll_id"])
xw_ct, diag_ct = build_crosswalk(poll_sources, ct_targets, match_ward=False, desc="polls -> whole CTs")
print(diag_ct)

mapped = mayor_polls[mayor_polls["poll_id"].isin(vsd["poll_id"])].rename(columns={"poll_id": "src_id"})
rederived = allocate(xw_ct, mapped[["src_id", "candidate", "votes"]]).reindex(ct["ctuid2021"])

rows = []
for cand, col in [("Chow Olivia", "a_chow_olivia2026"), ("Bradford Brad", "a_bradford_brad2026")]:
    mine, theirs = rederived[cand], dta.loc[rederived.index, col]
    rows.append({"candidate": cand, "this notebook": mine.sum(), ".dta": theirs.sum(),
                 "corr": np.corrcoef(mine, theirs)[0, 1], "mean |diff| (votes/CT)": (mine - theirs).abs().mean(),
                 "max |diff|": (mine - theirs).abs().max()})
pd.DataFrame(rows).round(3)

polls -> whole CTs:   0%|          | 0/1356 [00:00<?, ?it/s]

{'area_fallback': np.int64(3), 'nearest_fallback': 0, 'dropped': 0}


,candidate,this notebook,.dta,corr,mean |diff| (votes/CT),max |diff|
0,Chow Olivia,410363.0,410361.0,0.993,28.841,364.964
1,Bradford Brad,304832.0,304835.0,0.994,21.390,194.014


Citywide totals agree to within a few votes, so the same votes are being allocated; per-tract values differ by ~20–30 votes on average (correlation ≈ 0.99). The `.dta` was produced outside this repo, and this crosswalk uses 2021 DA adult-citizen counts, so the residual is the ancillary-weight and geometry difference, not a bug in either. **The mayoral GeoPackage uses the `.dta` values as given; the councillor GeoPackage uses this crosswalk.**

## 5. Mayoral GeoPackage (CTs)

Candidate votes come from the `.dta` (`a_<slug><year>`). For each year the chosen candidates are kept by name and everyone else is summed into `other`. Shares are fractions of the votes cast for any candidate in that CT. `votes_cast` is the `.dta`'s `voted<year>`, which also counts ballots with no valid mayoral vote. Each CT also gets the ward that holds most of it.

In [8]:
census_all = pd.read_parquet(CENSUS_PATH)
census = (census_all[census_all["year"] == CENSUS_YEAR].drop(columns="year")
          .rename(columns={"ct": "ctuid2021"}).set_index("ctuid2021"))
assert set(census.index) == set(ct["ctuid2021"])
census = census.astype("float64")
census_meta = json.load(open(CENSUS_META_PATH))

mayor = pd.DataFrame(index=ct["ctuid2021"])
for year, chosen in MAYOR_CANDIDATES.items():
    cols = {c: c[2:-4] for c in dta.columns if re.fullmatch(rf"a_.+{year}", c) and c != f"a_pop{year}"}
    votes = dta[list(cols)].rename(columns=cols).reindex(mayor.index).fillna(0.0)
    missing = set(chosen) - set(votes.columns)
    assert not missing, f"{year}: not in .dta: {missing}"
    total = votes.sum(axis=1)
    for slug in chosen:
        mayor[f"mayor{year}_{slug}_votes"] = votes[slug]
    mayor[f"mayor{year}_other_votes"] = total - votes[list(chosen)].sum(axis=1)
    for slug in [*chosen, "other"]:
        mayor[f"mayor{year}_{slug}_pct"] = mayor[f"mayor{year}_{slug}_votes"] / total.where(total > 0)
    mayor[f"mayor{year}_votes_candidates"] = total
    mayor[f"mayor{year}_votes_cast"] = dta[f"voted{year}"].reindex(mayor.index)
    print(year, f"{len(cols)} candidates in .dta; kept {len(chosen)} + other; "
                f"citywide: other = {mayor[f'mayor{year}_other_votes'].sum():,.0f} of {total.sum():,.0f}")

2023 102 candidates in .dta; kept 7 + other; citywide: other = 52,621 of 566,398
2026 53 candidates in .dta; kept 2 + other; citywide: other = 34,812 of 750,008


## 6. Split CTs by ward

Intersect the 585 CTs with the 25 wards. Shares are measured against the CT's area *inside* wards (a few lakefront CTs extend into the lake).

- Largest ward share ≥ 95% → CT is kept whole and assigned to that ward.
- Otherwise the CT is split into one piece per ward, except that a ward holding <5% of the CT gets no piece; its sliver is merged into the CT's largest piece so the map has no holes. Piece shares are then re-measured on the final piece geometries and sum to 1.

In [9]:
ov = gpd.overlay(ct, wards[["ward", "geometry"]], how="intersection", keep_geom_type=True)
ov["area_m2"] = ov.geometry.area
ov["share"] = ov["area_m2"] / ov.groupby("ctuid2021")["area_m2"].transform("sum")
top = ov.groupby("ctuid2021")["share"].max()
split_cts = set(top[top < UNSPLIT_THRESHOLD].index)
print(f"{len(ct) - len(split_cts)} CTs have >= {UNSPLIT_THRESHOLD:.0%} of their area in one ward; {len(split_cts)} are candidates for splitting")

ct_geom = ct.set_index("ctuid2021").geometry
pieces = []
for cid, g in ov.groupby("ctuid2021"):
    if cid not in split_cts:
        w = int(g.sort_values("share").iloc[-1]["ward"])
        pieces.append({"ctuid2021": cid, "ward": w, "geometry": ct_geom[cid], "is_split": False})
        continue
    g = g.sort_values("share", ascending=False)
    keep, drop = g[g["share"] >= MIN_PIECE_SHARE], g[g["share"] < MIN_PIECE_SHARE]
    geoms = {int(r.ward): r.geometry for r in keep.itertuples()}
    if len(drop):
        lead = int(keep.iloc[0]["ward"])
        geoms[lead] = union_all([geoms[lead], *drop["geometry"].values])
    for w, geom in geoms.items():
        pieces.append({"ctuid2021": cid, "ward": w, "geometry": geom, "is_split": len(geoms) > 1})

pieces = gpd.GeoDataFrame(pieces, crs=WORKING_EPSG)
pieces["piece_area_m2"] = pieces.geometry.area
pieces["area_share"] = pieces["piece_area_m2"] / pieces.groupby("ctuid2021")["piece_area_m2"].transform("sum")
pieces["piece_id"] = pieces["ctuid2021"] + "_w" + pieces["ward"].map("{:02d}".format)
pieces = pieces.merge(wards[["ward", "ward_name"]], on="ward").sort_values(["ward", "ctuid2021"]).reset_index(drop=True)
assert pieces["piece_id"].is_unique
assert np.allclose(pieces.groupby("ctuid2021")["area_share"].sum(), 1.0)

n_split = pieces.loc[pieces["is_split"], "ctuid2021"].nunique()
print(f"{len(pieces)} pieces: {n_split} CTs split into {int(pieces['is_split'].sum())} pieces, {len(pieces) - int(pieces['is_split'].sum())} CTs kept whole")
print("pieces per split CT:", pieces[pieces["is_split"]].groupby("ctuid2021").size().value_counts().to_dict())
pieces[pieces["is_split"]].head(6)[["piece_id", "ward", "area_share", "is_split"]]

540 CTs have >= 95% of their area in one ward; 45 are candidates for splitting


632 pieces: 45 CTs split into 92 pieces, 540 CTs kept whole
pieces per split CT: {2: 43, 3: 2}


,piece_id,ward,area_share,is_split
0,5350235.01_w01,1,0.114052,True
2,5350239.00_w01,1,0.673720,True
3,5350241.00_w01,1,0.136793,True
24,5350314.02_w01,1,0.091143,True
33,5350226.00_w02,2,0.373331,True
40,5350235.01_w02,2,0.885948,True


In [10]:
# dominant ward per CT, for the mayoral layer
dominant = (pieces.sort_values("area_share").groupby("ctuid2021").tail(1)
            .set_index("ctuid2021")[["ward", "area_share"]].rename(columns={"ward": "ward_dominant", "area_share": "ward_dominant_share"}))

## 7. Allocate 2026 councillor votes to CT×ward pieces

Sources are the 1,356 voting-subdivision polygons; each is only allowed to feed pieces in its own ward. The 97/98/99 buckets have no polygon: by default they are left out (same as the mayoral `.dta`). Set `INCLUDE_SPECIAL_BUCKETS = True` to spread each ward's bucket votes over that ward's pieces using the ward polygon as the source (population-weighted, no area fallback).

In [11]:
piece_targets = pieces.assign(tgt_id=pieces["piece_id"])
sources = vsd.assign(src_id=vsd["poll_id"])[["src_id", "ward", "geometry"]]
votes_src = council_polls[council_polls["poll_id"].isin(vsd["poll_id"])].rename(columns={"poll_id": "src_id"})

xw_piece, diag_piece = build_crosswalk(sources, piece_targets, match_ward=True, desc="polls -> CT x ward pieces")
print("diagnostics:", diag_piece)
allocated = allocate(xw_piece, votes_src[["src_id", "candidate", "votes"]])

if INCLUDE_SPECIAL_BUCKETS:
    bucket_votes = (council_polls[council_polls["sub"].isin(SPECIAL_BUCKETS)]
                    .assign(src_id=lambda d: d["ward"].map("B{:02d}".format))
                    .groupby(["src_id", "candidate"], as_index=False)["votes"].sum())
    bucket_src = wards.assign(src_id=wards["ward"].map("B{:02d}".format))[["src_id", "ward", "geometry"]]
    xw_bucket, diag_bucket = build_crosswalk(bucket_src, piece_targets, match_ward=True,
                                             allow_area_fallback=False, desc="ward buckets -> pieces")
    print("bucket diagnostics:", diag_bucket)
    allocated = allocated.add(allocate(xw_bucket, bucket_votes), fill_value=0.0)

polls -> CT x ward pieces:   0%|          | 0/1356 [00:00<?, ?it/s]

diagnostics: {'area_fallback': np.int64(3), 'nearest_fallback': 0, 'dropped': 0}


**Conservation checks.** For every ward and candidate, votes allocated to that ward's pieces must equal that ward's source votes, and a candidate must get no votes outside their own ward.

In [12]:
piece_ward = pieces.set_index("piece_id")["ward"]
cand_ward = council_polls.groupby("candidate")["ward"].agg(lambda s: s.unique().tolist())
assert (cand_ward.str.len() == 1).all(), "a candidate name appears in more than one ward"
cand_ward = cand_ward.str[0]

alloc_long = allocated.stack().rename("votes").reset_index().rename(columns={"tgt_id": "piece_id", "level_1": "candidate"})
alloc_long["ward"] = alloc_long["piece_id"].map(piece_ward)
alloc_long = alloc_long[alloc_long["votes"] > 0]
assert (alloc_long["candidate"].map(cand_ward) == alloc_long["ward"]).all(), "votes landed outside the candidate's ward"

if not INCLUDE_SPECIAL_BUCKETS:
    source_votes = council_polls[~council_polls["sub"].isin(SPECIAL_BUCKETS)]
else:
    source_votes = council_polls
by_candidate = pd.DataFrame({
    "source": source_votes.groupby("candidate")["votes"].sum(),
    "allocated": alloc_long.groupby("candidate")["votes"].sum(),
}).fillna(0.0)
assert ((by_candidate["source"] - by_candidate["allocated"]).abs() <= TOLERANCE * by_candidate["source"].clip(lower=1)).all()
print(f"all {len(by_candidate)} candidates conserved; {allocated.values.sum():,.1f} votes allocated to {len(pieces)} pieces "
      f"(source {source_votes['votes'].sum():,.0f})")

all 190 candidates conserved; 708,474.0 votes allocated to 632 pieces (source 708,474)


## 8. Census profile on the pieces

2021 census values for whole CTs. For a split CT each piece gets its **area share** of the CT's:

- **counts** (`*_abs`) and **denominators** (`den_*`) — multiplied by the share, so the pieces add back to the CT total;
- **shares** (`*_pct`), **z-scores** (`s_*`) and **nominal values** (medians, averages, density, distance, proximity indices) — copied unchanged, since area-scaling both numerator and denominator leaves a ratio untouched and non-additive measures have no area-proportional split.

`census_variable_dictionary.csv` records which rule applies to each column.

In [13]:
count_cols = [c for c in census.columns if c.endswith("_abs") or c.startswith("den_")]
copy_cols = [c for c in census.columns if c not in count_cols]
print(f"{len(count_cols)} count/denominator columns split by area share; {len(copy_cols)} ratio/z/nominal columns copied")

piece_census = census.loc[pieces["ctuid2021"]].reset_index(drop=True)
piece_census[count_cols] = piece_census[count_cols].mul(pieces["area_share"].values, axis=0)

check = piece_census[count_cols].groupby(pieces["ctuid2021"].values).sum(min_count=1)   # keep suppressed (NaN) CTs as NaN
assert np.allclose(check.values, census.loc[check.index, count_cols].values, equal_nan=True), "split counts do not add back to the CT totals"
print("pieces add back to the CT totals for every count column")

# data dictionary
by_id = {v["id"]: v for v in census_meta["variables"]}
dictionary = []
for c in census.columns:
    if c.endswith("_pct"):
        v = by_id.get(c, {}); role, label, group, rule = "share", v.get("label"), v.get("group"), "copied"
    elif c.endswith("_abs"):
        v = by_id.get(c[:-4] + "_pct", {}); role, group, rule = "count", v.get("group"), "split by area share"
        label = f"{v.get('label', c).removeprefix('% ')} (count)"
    elif c.startswith("den_"):
        v = next((x for x in census_meta["variables"] if x.get("den") == c), {})
        role, label, group, rule = "denominator", v.get("den_label", c), v.get("group"), "split by area share"
    elif c.startswith("s_"):
        v = by_id.get(c, {}); role, label, group, rule = "z-score", v.get("label"), "Standardised", "copied"
    else:
        v = next((x for x in census_meta["variables"] if x.get("nom") == c), {})
        role, label, group, rule = "nominal", v.get("label"), v.get("group", "Standardised"), "copied"
    dictionary.append({"column": c, "role": role, "label": label or c, "group": group, "unit": v.get("unit"), "split_rule": rule})
dictionary = pd.DataFrame(dictionary)
dictionary.to_csv(DICTIONARY_CSV, index=False)
dictionary["role"].value_counts().to_dict()

111 count/denominator columns split by area share; 125 ratio/z/nominal columns copied
pieces add back to the CT totals for every count column


{'share': 93, 'count': 93, 'denominator': 18, 'z-score': 16, 'nominal': 16}

## 9. Assemble and write the GeoPackages

**Councillor layer** (`councillor_ct_pieces`): one row per CT×ward piece with the piece's votes for every candidate in its ward (`votes_wNN_<name>`; other wards' candidates are empty), the total, and the winner / runner-up. Also written: `councillor_votes_long` (piece × candidate votes and shares, no geometry) and `wards`.

In [14]:
def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

cand_table = (council_polls.groupby("candidate")["ward"].first().rename("ward").reset_index()
              .assign(slug=lambda d: d["candidate"].map(slugify)))
cand_table["column"] = "votes_w" + cand_table["ward"].map("{:02d}".format) + "_" + cand_table["slug"]
assert cand_table["column"].is_unique

votes_wide = allocated.reindex(index=pieces["piece_id"], columns=cand_table["candidate"], fill_value=0.0)
# blank (NaN) for candidates who were not on the piece's ward ballot
in_ward = (pieces["ward"].values[:, None] == cand_table["ward"].values[None, :])
votes_wide = votes_wide.where(in_ward)
votes_wide.columns = cand_table["column"].values

ranked = votes_wide.set_axis(cand_table["candidate"].values, axis=1)
total = ranked.sum(axis=1, min_count=1)
order = np.argsort(-ranked.fillna(-1).values, axis=1)
names = ranked.columns.values
first, second = order[:, 0], order[:, 1]
rows = np.arange(len(ranked))

council = pieces[["piece_id", "ctuid2021", "ward", "ward_name", "is_split", "area_share"]].copy()
council["area_km2"] = pieces["piece_area_m2"].values / 1e6
council["votes_total"] = total.values
council["n_candidates"] = in_ward.sum(axis=1)
council["winner"] = names[first]
council["winner_votes"] = ranked.values[rows, first]
council["winner_pct"] = council["winner_votes"] / council["votes_total"].where(council["votes_total"] > 0)
council["runner_up"] = names[second]
council["runner_up_votes"] = ranked.values[rows, second]
council["runner_up_pct"] = council["runner_up_votes"] / council["votes_total"].where(council["votes_total"] > 0)
council["margin_pct"] = council["winner_pct"] - council["runner_up_pct"]
council = pd.concat([council.reset_index(drop=True), votes_wide.reset_index(drop=True), piece_census], axis=1)
council = gpd.GeoDataFrame(council, geometry=pieces.geometry.values, crs=WORKING_EPSG).to_crs(OUTPUT_EPSG)

long = (alloc_long.merge(cand_table[["candidate", "slug"]], on="candidate")
        .assign(votes_piece_total=lambda d: d["piece_id"].map(council.set_index("piece_id")["votes_total"]))
        .assign(pct=lambda d: d["votes"] / d["votes_piece_total"])
        .merge(pieces[["piece_id", "ctuid2021"]], on="piece_id")
        [["piece_id", "ctuid2021", "ward", "candidate", "slug", "votes", "pct"]]
        .sort_values(["ward", "ctuid2021", "votes"], ascending=[True, True, False]).reset_index(drop=True))

wards_out = wards.to_crs(OUTPUT_EPSG)

for p in (COUNCILLOR_GPKG, MAYOR_GPKG):
    p.unlink(missing_ok=True)
council.to_file(COUNCILLOR_GPKG, layer="councillor_ct_pieces", driver="GPKG")
wards_out.to_file(COUNCILLOR_GPKG, layer="wards", driver="GPKG")
pyogrio.write_dataframe(long, COUNCILLOR_GPKG, layer="councillor_votes_long", driver="GPKG", append=True)
print(f"{COUNCILLOR_GPKG.name}: councillor_ct_pieces {council.shape}, councillor_votes_long {long.shape}, wards {wards_out.shape}")

councillor_2026_ct.gpkg: councillor_ct_pieces (632, 443), councillor_votes_long (4801, 7), wards (25, 3)


**Mayoral layer** (`mayor_ct`): the 585 whole CTs with the 2023 and 2026 vote columns, the dominant ward, and the full 2021 census profile.

In [15]:
mayor_out = ct.set_index("ctuid2021")[["geometry"]].copy()
mayor_out["area_km2"] = mayor_out.geometry.area / 1e6
mayor_out = mayor_out.join(dominant).join(mayor).join(census)
mayor_out = gpd.GeoDataFrame(mayor_out.reset_index(), geometry="geometry", crs=WORKING_EPSG).to_crs(OUTPUT_EPSG)
mayor_out = mayor_out[["ctuid2021", "ward_dominant", "ward_dominant_share", "area_km2",
                       *mayor.columns, *census.columns, "geometry"]]
mayor_out.to_file(MAYOR_GPKG, layer="mayor_ct", driver="GPKG")
print(f"{MAYOR_GPKG.name}: mayor_ct {mayor_out.shape}")

mayor_2023_2026_ct.gpkg: mayor_ct (585, 267)


## 10. Final checks

In [16]:
# read back what was written
for path in (MAYOR_GPKG, COUNCILLOR_GPKG):
    print(path.name, pyogrio.list_layers(path).tolist())
m = gpd.read_file(MAYOR_GPKG, layer="mayor_ct")
c = gpd.read_file(COUNCILLOR_GPKG, layer="councillor_ct_pieces")
print("CRS:", m.crs.to_epsg(), c.crs.to_epsg(), "| all geometries valid:", bool(m.is_valid.all() and c.is_valid.all()))

print("\nMayor: citywide vote totals from the layer")
tot = m.filter(regex=r"^mayor\d{4}_.*_votes$").sum().round(0)
print(tot.to_string())

print("\nCouncillor: votes by ward vs. source (mapped subdivisions only"
      + (", plus 97-99 buckets)" if INCLUDE_SPECIAL_BUCKETS else ")"))
ward_check = pd.DataFrame({
    "layer": c.groupby("ward")["votes_total"].sum(),
    "source": source_votes.groupby("ward")["votes"].sum(),
})
assert np.allclose(ward_check["layer"], ward_check["source"])
print(ward_check.round(0).astype(int).T.to_string())

print("\nWinners:", c.loc[c.groupby("ward")["votes_total"].idxmax()].shape[0], "wards;",
      "pieces with winner margin < 5 pts:", int((c["margin_pct"] < 0.05).sum()))

mayor_2023_2026_ct.gpkg [['mayor_ct', 'MultiPolygon']]
councillor_2026_ct.gpkg [['councillor_ct_pieces', 'MultiPolygon'], ['wards', 'Polygon'], ['councillor_votes_long', None]]


CRS: 4326 4326 | all geometries valid: True

Mayor: citywide vote totals from the layer
mayor2023_chow_olivia_votes      194656.0
mayor2023_bailao_ana_votes       212730.0
mayor2023_saunders_mark_votes     43798.0
mayor2023_furey_anthony_votes     26524.0
mayor2023_hunter_mitzie_votes     14421.0
mayor2023_brown_chloe_votes       15588.0
mayor2023_bradford_brad_votes      6060.0
mayor2023_other_votes             52621.0
mayor2026_chow_olivia_votes      410361.0
mayor2026_bradford_brad_votes    304835.0
mayor2026_other_votes             34812.0

Councillor: votes by ward vs. source (mapped subdivisions only)
ward       1      2      3      4      5      6      7      8      9      10     11     12     13     14     15     16     17     18     19     20     21     22     23     24     25
layer   22291  33003  40641  36150  26794  24788  20220  30631  32452  36700  31595  34129  31195  33865  27846  22144  23426  24220  34949  26864  24698  20967  19522  22054  27330
source  22291  33003 